In [61]:
from astropy.time import Time
from astroquery.jplhorizons import Horizons
from astroquery.mpc import MPC
from io import StringIO
from sora import Body
from sora import EphemPlanete

In [56]:
# Hiperparámetros
rock = 'Silesia'
obs = 'W63'
epoch = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1h'
}

In [57]:
# Bajamos efemérides del asteroide con horizons
body_jpl = Horizons(id=rock, location=obs, epochs=epoch)
eph_jpl = body_jpl.ephemerides()
data_jpl = eph_jpl['datetime_jd', 'RA', 'DEC', 'delta'] # Datos que pide SORA en objeto efemérides
sigma_jpl = eph_jpl['RA_3sigma', 'DEC_3sigma'] # incertidumbre 3sigma para la ascención recta y declinación en ese instante
error_jpl = eph_jpl['SMAA_3sigma', 'SMIA_3sigma', 'Theta_3sigma'] # característica del elipse de error (matriz de covarianza diagonalizada)

In [58]:
# Bajamos efemérides del asteroide con MPCES
body_mpc = MPC.query_object('asteroid', name=rock)
eph_mpc = MPC.get_ephemeris(rock, step=epoch['step'], start=epoch['start'], number=49, location=obs)
eph_mpc['Date_jd'] = Time(eph_mpc['Date']).jd # Las fechas tienen que estar en formato juliano
data_mpc = eph_mpc['Date_jd', 'RA', 'Dec', 'Delta'] # Datos que pide SORA en objeto efemérides
error_mpc = eph_mpc['Uncertainty 3sig', 'Unc. P.A.'] # características del error principal (componente principal)

In [59]:
# Función para convertir tabla astropy de query a ephem planete de sora (ya permite predicciones con cualquier query)
def ephem_sora(name, data):
  df = data.to_pandas()
  buffer = StringIO()
  df.to_csv(buffer, sep=' ', header=False, index=False, float_format='%.12f')
  buffer.seek(0)
  return EphemPlanete(ephem=buffer, name=name)

In [60]:
# Instanciamos objeto de efemérides de JPL y MPC
eph_jpl_sora = ephem_sora(rock, data_jpl)
eph_mpc_sora = ephem_sora(rock, data_mpc)

# Instanciamos objeto menor con SORA utilizando los datos de jpl
sora_body_jpl = Body(rock, ephem=eph_jpl_sora)
sora_body_mpc = Body(rock, ephem=eph_mpc_sora)

Obtaining data for Silesia from SBDB


d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\body\meta.py:372: UserWarning: spkid is different in Body (20000257) and EphemPlanete (None). Body's spkid will have higher priority
  warnings.warn('spkid is different in {0} ({1}) and {2} ({3}). {0}\'s spkid will have higher priority'.format(


Obtaining data for Silesia from SBDB


d:\Users\andre\Escritorio\Universal\Code\Python\projects\occ\.venv\Lib\site-packages\sora\body\meta.py:372: UserWarning: spkid is different in Body (20000257) and EphemPlanete (None). Body's spkid will have higher priority
  warnings.warn('spkid is different in {0} ({1}) and {2} ({3}). {0}\'s spkid will have higher priority'.format(
